# Student Budget Assistant
**ISYS2001 – Assessment 2: Programming Project**

A small finance assistant that helps an international student in Perth understand where their money goes each month, check their spending against a budget they set, plan a savings goal, and ask questions about their own spending in plain English.

**Notebook map**
1. Step 1 – Understand the problem
2. Step 2 – Inputs and outputs
3. Step 3 – Worked example by hand
4. Step 4 – Pseudocode
5. Step 5 – Python code (setup → data loading → analysis → custom tools → Gemini assistant → Gradio app)
6. Step 6 – Testing (normal, edge and error cases, plus a bug I found and fixed)

## Step 1 – Understand the problem

**User.** An international university student living in Perth. They pay rent weekly, work a casual job, and spend on groceries, transport, eating out and subscriptions. Their money arrives weekly but their spending is spread across many small purchases, so by the end of the month they often do not know where it went.

**Business problem.** Small, frequent purchases (food delivery, subscriptions) add up without the student noticing. Banking apps show a list of transactions, but they do not say *"you are 40% over your eating-out budget"* or *"at this rate you will reach your $1,200 flight-home goal in 9 weeks"*. The student needs something that turns their own transaction list into decisions.

**What the assistant does.**
- Reads the student's own transaction CSV (exported from their bank, or typed up).
- Summarises income, spending and spending by category.
- **Budget checker (custom tool):** compares spending in each category against limits the student sets and flags *Under / Close to limit / Over*.
- **Savings goal planner (custom tool):** works out how much to save per week to hit a goal by a deadline, and whether their current surplus is enough.
- **Chat assistant:** a Gemini-powered "budget coach" that answers questions using the student's real numbers, not generic advice.

**Out of scope.** It does not give investment, tax or credit advice, and it does not connect to a bank.

**Success looks like:** the student uploads their CSV and within a minute can answer *"Where did my money go, which category is out of control, and how much should I put aside each week?"*

## Step 2 – Inputs and outputs

| Part | Inputs | Outputs |
|---|---|---|
| Load data | A CSV file with columns `date, description, category, amount` | A list of clean transaction dictionaries + a list of warnings about rows that were skipped or fixed |
| Summary | The clean transactions | A dictionary: total income, total spending, net (income − spending), spending per category, biggest category, number of transactions |
| **Budget checker** | Spending per category (dict) + budget limits typed as text, e.g. `Groceries=250, Eating Out=100` | For each budgeted category: spent, limit, remaining, % used, status (`Under`, `Close to limit` ≥ 80%, `Over`) |
| **Savings planner** | Goal amount, amount already saved, number of weeks, current weekly surplus | Amount still needed, saving needed per week, whether the current surplus is enough, and how many weeks it would take at the current rate |
| Chat assistant | The user's question + a text summary of their data + chat history | A reply from Gemini in the "budget coach" persona |

**Rules I decided on**
- `Income` rows count as income; every other category counts as spending.
- A negative amount in a spending category is a refund and reduces that category.
- Amounts may contain `$` or commas (`$1,200.50`), which are cleaned before converting.
- Rows with a missing or unreadable amount are skipped and reported, not silently dropped.
- A blank category becomes `Uncategorised`. Categories are tidied to Title Case so `groceries` and `Groceries` are the same.
- Budget limits must be numbers ≥ 0. Savings inputs must be numbers, goal > 0, weeks > 0.

## Step 3 – Worked example by hand

A tiny made-up month with five rows:

| date | description | category | amount |
|---|---|---|---|
| 01/09 | Casual job pay | Income | 500.00 |
| 02/09 | Coles | Groceries | 60.00 |
| 03/09 | Uber Eats | Eating Out | 45.00 |
| 05/09 | Aldi | Groceries | 40.00 |
| 06/09 | Uber Eats | Eating Out | 70.00 |

**Summary by hand**
- Total income = 500.00
- Spending: Groceries = 60 + 40 = **100.00**, Eating Out = 45 + 70 = **115.00**
- Total spending = 100 + 115 = **215.00**
- Net = 500 − 215 = **285.00**
- Biggest category = Eating Out (115.00)

**Budget check by hand** with limits `Groceries=150, Eating Out=100`
- Groceries: 100 / 150 = 66.7% → remaining 50.00 → **Under**
- Eating Out: 115 / 100 = 115% → remaining −15.00 → **Over**

If Groceries had been 125: 125 / 150 = 83.3% → ≥ 80% → **Close to limit**.

**Savings plan by hand** – goal $1,200 flight home, already saved $300, 10 weeks, current weekly surplus $80
- Still needed = 1200 − 300 = **900**
- Needed per week = 900 / 10 = **90.00**
- 80 < 90 → **not on track**, short by 10.00 per week
- Weeks at current rate = 900 / 80 = 11.25 → round **up** to **12 weeks** (you cannot finish in a quarter of a week)

These numbers become the expected values in my tests in Step 6.

## Step 4 – Pseudocode

```
FUNCTION clean_amount(text)
    remove spaces, "$" and "," from text
    IF text is empty: RAISE error "missing amount"
    convert text to a number (raise error if this fails)
    RETURN number

FUNCTION load_transactions(file)
    read CSV with pandas
    IF any required column is missing: RAISE error listing the missing columns
    transactions = empty list, warnings = empty list
    FOR each row (with its row number):
        TRY amount = clean_amount(row amount)
        EXCEPT: add warning "row N skipped: bad amount"; CONTINUE to next row
        category = row category stripped and in Title Case, or "Uncategorised" if blank
        add {date, description, category, amount} to transactions
    RETURN transactions, warnings

FUNCTION summarise(transactions)
    income = 0, by_category = empty dict
    FOR each transaction:
        IF category is "Income": income = income + amount
        ELSE: by_category[category] = by_category.get(category, 0) + amount
    spending = sum of by_category values
    biggest = category with the largest value (or None if no spending)
    RETURN dict of income, spending, net = income − spending, by_category, biggest, count

FUNCTION parse_budgets(text)            # "Groceries=250, Eating Out=100"
    budgets = empty dict
    FOR each piece split by ",":
        skip blank pieces
        IF piece has no "=": RAISE error
        split into name and value; tidy name to Title Case
        value = number(value); IF value < 0: RAISE error
        budgets[name] = value
    IF budgets is empty: RAISE error
    RETURN budgets

FUNCTION check_budget(spending_by_category, budgets)     # CUSTOM TOOL 1
    results = empty list
    FOR each category, limit in budgets:
        spent = spending_by_category.get(category, 0)
        remaining = limit − spent
        percent = spent / limit × 100   (if limit is 0: 0% when nothing spent, otherwise infinite → Over)
        IF spent > limit: status = "Over"
        ELSE IF percent ≥ 80: status = "Close to limit"
        ELSE: status = "Under"
        add {category, spent, limit, remaining, percent, status} to results
    sort results so the worst (highest percent) is first
    RETURN results

FUNCTION plan_savings(goal, saved, weeks, weekly_surplus)   # CUSTOM TOOL 2
    check: goal > 0, saved ≥ 0, weeks > 0 and whole, surplus is a number
    needed = max(goal − saved, 0)
    IF needed is 0: RETURN "goal already reached"
    per_week = needed / weeks
    on_track = weekly_surplus ≥ per_week
    IF weekly_surplus > 0: weeks_at_current_rate = round UP (needed / weekly_surplus)
    ELSE: weeks_at_current_rate = None (never at this rate)
    RETURN dict of all of the above

CLASS BudgetCoach
    holds: api key, model name, system instruction, chat history
    METHOD ask(question, data_context)
        IF question is blank: RETURN a polite prompt to ask something
        build request = system instruction + history + (data_context + question)
        TRY send POST request to Gemini with a timeout
        EXCEPT network error / bad status: RETURN friendly error message
        pull the reply text out of the response
        add question and reply to history
        RETURN reply

GRADIO APP
    Tab 1 Upload: file → load + summarise → show summary, category table, warnings
    Tab 2 Budget: budget text → check_budget on the uploaded data → table
    Tab 3 Savings: four numbers → plan_savings → message
    Tab 4 Chat: question → BudgetCoach.ask with the uploaded data → reply
    Every handler catches errors and shows a message instead of crashing
```

## Step 5 – Convert to Python

### 5.1 Setup

The Gemini API key is **not** written in this notebook. In Colab, click the 🔑 **Secrets** icon in the left sidebar, add a secret named `GEMINI_API_KEY`, paste your free key from [Google AI Studio](https://aistudio.google.com/apikey), and turn on *Notebook access*. Outside Colab, set an environment variable with the same name.

In [ ]:
# Gradio is not pre-installed in Colab; pandas and requests already are.
!pip install -q gradio

In [ ]:
import math
import os

import pandas as pd
import requests
import gradio as gr

In [ ]:
def get_api_key():
    # Try Colab Secrets first, then an environment variable. Never hard-code the key.
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key
    except Exception:
        pass  # not running in Colab, or the secret has not been added
    return os.environ.get("GEMINI_API_KEY")


GEMINI_API_KEY = get_api_key()
GEMINI_MODEL = "gemini-flash-latest"

if GEMINI_API_KEY:
    print("Gemini API key found.")
else:
    print("No Gemini API key found - the chat tab will explain how to add one. Everything else still works.")

In [ ]:
# Sample data: if the repository's data folder is not next to the notebook (e.g. a fresh Colab),
# download the sample files from GitHub. Change GITHUB_RAW to your own repository.
GITHUB_RAW = "https://raw.githubusercontent.com/YOUR-USERNAME/student-budget-assistant/main/data/"
SAMPLE_FILES = ["sample_transactions.csv", "second_student.csv", "messy_transactions.csv"]

os.makedirs("data", exist_ok=True)
for name in SAMPLE_FILES:
    path = os.path.join("data", name)
    if not os.path.exists(path):
        try:
            response = requests.get(GITHUB_RAW + name, timeout=10)
            response.raise_for_status()
            with open(path, "w") as f:
                f.write(response.text)
            print("Downloaded", name)
        except requests.RequestException as error:
            print("Could not download", name, "-", error)
    else:
        print("Found", path)

### 5.2 Loading and cleaning the user's data

In [ ]:
REQUIRED_COLUMNS = ["date", "description", "category", "amount"]


def clean_amount(value):
    # Turn values like "$1,200.50", " 45 " or 60 into a float. Raises ValueError if it can't.
    text = str(value).strip().replace("$", "").replace(",", "")
    if text == "" or text.lower() == "nan":
        raise ValueError("missing amount")
    return float(text)  # float() itself raises ValueError for text like "abc"


def clean_category(value):
    # Tidy a category name: strip spaces, Title Case, and give blanks a name.
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return "Uncategorised"
    return text.title()


def load_transactions(file_path):
    # Read a CSV and return (list of transaction dicts, list of warning strings).
    try:
        df = pd.read_csv(file_path, dtype=str)  # read everything as text, I clean it myself
    except FileNotFoundError:
        raise ValueError("I couldn't find that file.")
    except pd.errors.EmptyDataError:
        raise ValueError("The file is empty.")
    except pd.errors.ParserError:
        raise ValueError("That doesn't look like a valid CSV file.")

    df.columns = [str(col).strip().lower() for col in df.columns]
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError("The CSV is missing these columns: " + ", ".join(missing)
                         + ". Expected: " + ", ".join(REQUIRED_COLUMNS))

    transactions = []
    warnings = []
    for index, row in df.iterrows():
        row_number = index + 2  # +2 because of the header row and Python counting from 0
        try:
            amount = clean_amount(row["amount"])
        except ValueError:
            warnings.append(f"Row {row_number} skipped: amount '{row['amount']}' is not a number.")
            continue

        category = clean_category(row["category"])
        if category == "Uncategorised":
            warnings.append(f"Row {row_number}: no category, filed under 'Uncategorised'.")

        transactions.append({
            "date": str(row["date"]).strip(),
            "description": str(row["description"]).strip(),
            "category": category,
            "amount": amount,
        })

    if len(transactions) == 0:
        raise ValueError("No usable transactions were found in the file.")
    return transactions, warnings


transactions, warnings = load_transactions("data/sample_transactions.csv")
print(len(transactions), "transactions loaded,", len(warnings), "warnings")
transactions[:3]

### 5.3 Summarising spending

I used my own dictionary-and-loop logic here instead of `groupby`, so I can walk through exactly how each total is built.

In [ ]:
def summarise(transactions):
    # Build totals from a list of transaction dicts.
    income = 0.0
    by_category = {}
    for t in transactions:
        if t["category"] == "Income":
            income += t["amount"]
        else:
            by_category[t["category"]] = by_category.get(t["category"], 0.0) + t["amount"]

    spending = sum(by_category.values())

    biggest = None
    for category, total in by_category.items():
        if biggest is None or total > by_category[biggest]:
            biggest = category

    return {
        "income": round(income, 2),
        "spending": round(spending, 2),
        "net": round(income - spending, 2),
        "by_category": {cat: round(total, 2) for cat, total in by_category.items()},
        "biggest_category": biggest,
        "count": len(transactions),
    }


summary = summarise(transactions)
summary

### 5.4 Custom tool 1 – Budget checker

In [ ]:
CLOSE_TO_LIMIT_PERCENT = 80


def parse_budgets(text):
    # Turn "Groceries=250, Eating Out=100" into {"Groceries": 250.0, "Eating Out": 100.0}.
    if text is None or str(text).strip() == "":
        raise ValueError("Please enter at least one budget, e.g. Groceries=250, Eating Out=100")

    budgets = {}
    for piece in str(text).split(","):
        piece = piece.strip()
        if piece == "":
            continue  # allows a trailing comma
        if "=" not in piece:
            if piece.isdigit():
                # e.g. "Rent=1,000" splits into "Rent=1" and "000"
                raise ValueError("Don't use commas inside numbers - write 1000, not 1,000.")
            raise ValueError(f"'{piece}' needs an equals sign, e.g. Groceries=250")
        name, value = piece.split("=", 1)
        name = clean_category(name)
        try:
            limit = clean_amount(value)
        except ValueError:
            raise ValueError(f"The budget for {name} ('{value.strip()}') is not a number.")
        if limit < 0:
            raise ValueError(f"The budget for {name} can't be negative.")
        budgets[name] = limit

    if not budgets:
        raise ValueError("Please enter at least one budget, e.g. Groceries=250")
    return budgets


def check_budget(spending_by_category, budgets):
    # Compare spending with each budget limit. Returns a list of result dicts, worst first.
    results = []
    for category, limit in budgets.items():
        spent = spending_by_category.get(category, 0.0)
        remaining = limit - spent

        if limit > 0:
            percent = spent / limit * 100
        elif spent > 0:
            percent = math.inf  # a $0 budget with any spending is infinitely over
        else:
            percent = 0.0

        if spent > limit:
            status = "Over"
        elif percent >= CLOSE_TO_LIMIT_PERCENT:
            status = "Close to limit"
        else:
            status = "Under"

        results.append({
            "category": category,
            "spent": round(spent, 2),
            "limit": round(limit, 2),
            "remaining": round(remaining, 2),
            "percent_used": round(percent, 1) if percent != math.inf else percent,
            "status": status,
        })

    results.sort(key=lambda r: r["percent_used"], reverse=True)
    return results


budgets = parse_budgets("Groceries=300, Eating Out=120, Subscriptions=20, Transport=100")
pd.DataFrame(check_budget(summary["by_category"], budgets))

### 5.5 Custom tool 2 – Savings goal planner

In [ ]:
def to_number(value, name):
    # Convert a user-entered value to float with a friendly error message.
    try:
        return clean_amount(value)
    except ValueError:
        raise ValueError(f"{name} must be a number.")


def plan_savings(goal, saved, weeks, weekly_surplus):
    # Work out what it takes to reach a savings goal. Returns a dict.
    goal = to_number(goal, "Goal amount")
    saved = to_number(saved, "Amount already saved")
    weeks = to_number(weeks, "Number of weeks")
    weekly_surplus = to_number(weekly_surplus, "Weekly surplus")

    if goal <= 0:
        raise ValueError("Goal amount must be more than 0.")
    if saved < 0:
        raise ValueError("Amount already saved can't be negative.")
    if weeks <= 0 or weeks != int(weeks):
        raise ValueError("Number of weeks must be a whole number above 0.")

    needed = max(goal - saved, 0.0)
    if needed == 0:
        return {"needed": 0.0, "per_week": 0.0, "on_track": True,
                "shortfall_per_week": 0.0, "weeks_at_current_rate": 0, "already_reached": True}

    per_week = needed / weeks
    on_track = weekly_surplus >= per_week
    if weekly_surplus > 0:
        weeks_at_current_rate = math.ceil(round(needed / weekly_surplus, 6))
    else:
        weeks_at_current_rate = None  # never, at the current rate

    return {
        "needed": round(needed, 2),
        "per_week": round(per_week, 2),
        "on_track": on_track,
        "shortfall_per_week": round(max(per_week - weekly_surplus, 0), 2),
        "weeks_at_current_rate": weeks_at_current_rate,
        "already_reached": False,
    }


def weekly_surplus_from_summary(summary, days_in_period=30):
    # Estimate how much is left over per week from the uploaded month.
    return round(summary["net"] / days_in_period * 7, 2)


plan_savings(1200, 300, 10, weekly_surplus_from_summary(summary))

### 5.6 The budget coach (Gemini)

I call the Gemini REST API with `requests` (the same way we called APIs in the unit) rather than an SDK, so every part of the request is visible. The user's real numbers are turned into a short text block and sent with each question, which is what grounds the answers in their data.

In [ ]:
SYSTEM_INSTRUCTION = '''You are "Penny", a friendly budget coach for university students in Australia.
Your purpose: help the student understand and improve their own spending, using ONLY the figures
given to you in the DATA section of each message. Amounts are Australian dollars.

Rules:
- Base every number you mention on the DATA. Never invent transactions or totals.
- If no data has been uploaded, say so and ask the student to upload their CSV in the first tab.
- Keep answers short (under 150 words), practical and encouraging. Suggest one or two concrete actions.
- You are not a licensed financial adviser. Do not give investment, tax, credit or legal advice;
  for those, suggest the student talks to Curtin's student finance services or a licensed adviser.
- If the question is not about personal finance or budgeting, politely say that is outside what
  you help with and steer back to their budget.
- If the question is unclear, ask one short clarifying question.'''


def build_data_context(summary, budget_results=None):
    # Turn the user's analysed data into plain text the model can read.
    if summary is None:
        return "DATA: No data uploaded yet."

    lines = ["DATA (this month):",
             f"- Income: ${summary['income']:.2f}",
             f"- Spending: ${summary['spending']:.2f}",
             f"- Net (income - spending): ${summary['net']:.2f}",
             f"- Biggest spending category: {summary['biggest_category']}",
             "- Spending by category:"]
    for category, total in sorted(summary["by_category"].items(), key=lambda item: item[1], reverse=True):
        lines.append(f"    {category}: ${total:.2f}")

    if budget_results:
        lines.append("- Budget check:")
        for r in budget_results:
            lines.append(f"    {r['category']}: spent ${r['spent']:.2f} of ${r['limit']:.2f} ({r['status']})")
    return "\n".join(lines)


class BudgetCoach:
    # A small chat assistant that remembers the conversation and talks to Gemini.

    def __init__(self, api_key, model=GEMINI_MODEL, system_instruction=SYSTEM_INSTRUCTION):
        self.api_key = api_key
        self.model = model
        self.system_instruction = system_instruction
        self.history = []  # list of {"role": "user"/"model", "parts": [{"text": ...}]}

    def reset(self):
        self.history = []

    def build_request(self, question, data_context):
        # The data goes with the newest question so the model always sees the latest numbers.
        user_turn = {"role": "user", "parts": [{"text": data_context + "\n\nQUESTION: " + question}]}
        return {
            "system_instruction": {"parts": [{"text": self.system_instruction}]},
            "contents": self.history + [user_turn],
        }

    def ask(self, question, data_context):
        if question is None or question.strip() == "":
            return "Ask me something about your spending, e.g. 'Where can I cut back this month?'"
        if not self.api_key:
            return ("I can't reach the language model yet: add a GEMINI_API_KEY secret in Colab "
                    "(key icon on the left), then re-run the setup cell.")

        url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent"
        try:
            response = requests.post(url,
                                     headers={"x-goog-api-key": self.api_key},
                                     json=self.build_request(question, data_context),
                                     timeout=30)
        except requests.RequestException:
            return "I couldn't connect to Gemini. Check your internet connection and try again."

        if response.status_code == 429:
            return "Gemini's free tier limit was reached. Wait a minute and try again."
        if response.status_code != 200:
            return f"Gemini returned an error ({response.status_code}). Check your API key and model name."

        try:
            reply = response.json()["candidates"][0]["content"]["parts"][0]["text"]
        except (KeyError, IndexError, ValueError):
            return "Gemini sent back an empty answer (it may have been blocked). Try rephrasing."

        # Store the plain question (without the data block) to keep the history short.
        self.history.append({"role": "user", "parts": [{"text": question}]})
        self.history.append({"role": "model", "parts": [{"text": reply}]})
        return reply

In [ ]:
# Quick live check (needs the API key). The answer should mention the user's real figures.
coach = BudgetCoach(GEMINI_API_KEY)
print(coach.ask("Which category should I cut back on first?", build_data_context(summary)))

### 5.7 The Gradio interface

Each tab calls one of the functions above. Every handler catches `ValueError` and shows the message to the user, so bad input produces an explanation instead of a crash. `gr.State` keeps each user's own data separate.

In [ ]:
def money(x):
    # 1234.5 -> "$1,234.50", -464 -> "-$464.00"
    sign = "-" if x < 0 else ""
    return f"{sign}${abs(x):,.2f}"


def handle_upload(file):
    # Tab 1: load the CSV and return (summary text, category table, warnings, state).
    if file is None:
        return "Please upload a CSV file.", None, "", None
    path = file if isinstance(file, str) else file.name
    try:
        transactions, warnings = load_transactions(path)
    except ValueError as error:
        return f"**Problem with the file:** {error}", None, "", None

    s = summarise(transactions)
    text = (f"### Your month at a glance\n"
            f"- **Income:** {money(s['income'])}\n"
            f"- **Spending:** {money(s['spending'])}\n"
            f"- **Left over:** {money(s['net'])}\n"
            f"- **Biggest category:** {s['biggest_category']}\n"
            f"- **Transactions read:** {s['count']}")
    table = pd.DataFrame(
        [{"Category": c, "Spent": money(v),
          "% of spending": f"{(v / s['spending'] * 100) if s['spending'] else 0:.1f}%"}
         for c, v in sorted(s["by_category"].items(), key=lambda item: item[1], reverse=True)])
    warning_text = "\n".join(warnings) if warnings else "No problems found in the file."
    return text, table, warning_text, s


def handle_budget(budget_text, summary_state):
    # Tab 2: run the budget checker on the uploaded data.
    if summary_state is None:
        return "Upload your transactions in the first tab first.", None, None
    try:
        budgets = parse_budgets(budget_text)
    except ValueError as error:
        return f"**Problem with your budgets:** {error}", None, None

    results = check_budget(summary_state["by_category"], budgets)
    icons = {"Over": "🔴 Over", "Close to limit": "🟠 Close to limit", "Under": "🟢 Under"}
    table = pd.DataFrame([{
        "Category": r["category"], "Spent": money(r["spent"]), "Budget": money(r["limit"]),
        "Remaining": money(r["remaining"]),
        "% used": "∞" if r["percent_used"] == math.inf else f"{r['percent_used']}%",
        "Status": icons[r["status"]]} for r in results])
    over = [r["category"] for r in results if r["status"] == "Over"]
    unknown = [c for c in budgets if c not in summary_state["by_category"]]
    message = f"**Over budget:** {', '.join(over)}" if over else "**Nice work - nothing is over budget.**"
    if unknown:
        message += f"\n\nNo spending found for: {', '.join(unknown)} (check the spelling matches your CSV)."
    return message, table, results


def handle_savings(goal, saved, weeks, surplus):
    # Tab 3: savings goal planner.
    try:
        p = plan_savings(goal, saved, weeks, surplus)
    except ValueError as error:
        return f"**Problem:** {error}"
    if p["already_reached"]:
        return "🎉 You've already reached this goal!"
    text = (f"- You still need **{money(p['needed'])}**.\n"
            f"- To finish in {int(float(weeks))} weeks, save **{money(p['per_week'])} per week**.\n")
    if p["on_track"]:
        text += f"- ✅ Your current surplus covers it. At that rate you'd finish in about **{p['weeks_at_current_rate']} weeks**."
    elif p["weeks_at_current_rate"] is None:
        text += "- ❌ You're not saving anything at the moment, so the goal won't be reached without cutting spending."
    else:
        text += (f"- ❌ You're **{money(p['shortfall_per_week'])} per week short**. "
                 f"At your current rate it would take **{p['weeks_at_current_rate']} weeks**.")
    return text


def fill_surplus(summary_state):
    # Pre-fill the weekly surplus from the uploaded month.
    if summary_state is None:
        return 0
    return weekly_surplus_from_summary(summary_state)


def handle_chat(message, chat_history, summary_state, budget_state, coach_state):
    # Tab 4: chat. Each browser session gets its own BudgetCoach (its own history).
    if coach_state is None:
        coach_state = BudgetCoach(GEMINI_API_KEY)
    reply = coach_state.ask(message, build_data_context(summary_state, budget_state))
    chat_history = (chat_history or []) + [{"role": "user", "content": message},
                                           {"role": "assistant", "content": reply}]
    return "", chat_history, coach_state


def clear_chat(coach_state):
    if coach_state is not None:
        coach_state.reset()
    return [], coach_state


with gr.Blocks(title="Student Budget Assistant") as app:
    gr.Markdown("# 💰 Student Budget Assistant\nUpload your transactions, set your budgets, plan a goal, and ask Penny, your budget coach.")
    summary_state = gr.State(None)
    budget_state = gr.State(None)
    coach_state = gr.State(None)

    with gr.Tab("1. Upload"):
        gr.Markdown("CSV columns needed: `date, description, category, amount`. Use the category `Income` for money coming in.")
        file_in = gr.File(label="Transactions CSV", file_types=[".csv"])
        gr.Examples([["data/sample_transactions.csv"], ["data/second_student.csv"], ["data/messy_transactions.csv"]],
                    inputs=file_in)
        summary_out = gr.Markdown()
        category_out = gr.Dataframe(label="Spending by category")
        warnings_out = gr.Textbox(label="File check", lines=3)

    with gr.Tab("2. Budget checker"):
        budget_in = gr.Textbox(label="Your monthly budgets",
                               value="Groceries=300, Eating Out=120, Subscriptions=20, Transport=100")
        budget_btn = gr.Button("Check my budget", variant="primary")
        budget_msg = gr.Markdown()
        budget_out = gr.Dataframe(label="Budget check")

    with gr.Tab("3. Savings goal"):
        with gr.Row():
            goal_in = gr.Number(label="Goal amount ($)", value=1200)
            saved_in = gr.Number(label="Already saved ($)", value=300)
        with gr.Row():
            weeks_in = gr.Number(label="Weeks until I need it", value=10, precision=0)
            surplus_in = gr.Number(label="What I have left over per week ($)", value=0)
        savings_btn = gr.Button("Plan my savings", variant="primary")
        savings_out = gr.Markdown()

    with gr.Tab("4. Ask Penny"):
        chatbot = gr.Chatbot(label="Penny, your budget coach", height=380)
        chat_in = gr.Textbox(label="Your question", placeholder="e.g. Where is most of my money going?")
        with gr.Row():
            send_btn = gr.Button("Send", variant="primary")
            clear_btn = gr.Button("Clear chat")

    file_in.change(handle_upload, inputs=file_in,
                   outputs=[summary_out, category_out, warnings_out, summary_state])
    summary_state.change(fill_surplus, inputs=summary_state, outputs=surplus_in)
    budget_btn.click(handle_budget, inputs=[budget_in, summary_state],
                     outputs=[budget_msg, budget_out, budget_state])
    savings_btn.click(handle_savings, inputs=[goal_in, saved_in, weeks_in, surplus_in], outputs=savings_out)
    chat_args = dict(fn=handle_chat, inputs=[chat_in, chatbot, summary_state, budget_state, coach_state],
                     outputs=[chat_in, chatbot, coach_state])
    send_btn.click(**chat_args)
    chat_in.submit(**chat_args)
    clear_btn.click(clear_chat, inputs=coach_state, outputs=[chatbot, coach_state])

In [ ]:
app.launch(debug=False)

## Step 6 – Testing

The expected values come from the worked example in Step 3, so the tests check specific numbers, not just that something came back. A small helper runs each test and prints PASS/FAIL so one failure doesn't hide the rest.

In [ ]:
results_log = []

def run_test(name, test_function):
    try:
        test_function()
        results_log.append((name, "PASS"))
    except AssertionError as error:
        results_log.append((name, f"FAIL {error}"))
    except Exception as error:  # a crash inside a test is also a failure
        results_log.append((name, f"FAIL crashed: {type(error).__name__}: {error}"))


def expect_error(function, *args):
    # Returns True if calling function(*args) raises ValueError.
    try:
        function(*args)
    except ValueError:
        return True
    return False


# The five-row month from Step 3, written as a CSV file
with open("data/worked_example.csv", "w") as f:
    f.write("date,description,category,amount\n"
            "2026-09-01,Casual job pay,Income,500.00\n"
            "2026-09-02,Coles,Groceries,60.00\n"
            "2026-09-03,Uber Eats,Eating Out,45.00\n"
            "2026-09-05,Aldi,Groceries,40.00\n"
            "2026-09-06,Uber Eats,Eating Out,70.00\n")
example, _ = load_transactions("data/worked_example.csv")
example_summary = summarise(example)

### Normal cases

In [ ]:
def test_summary_matches_hand_calculation():
    assert example_summary["income"] == 500.00, example_summary["income"]
    assert example_summary["spending"] == 215.00, example_summary["spending"]
    assert example_summary["net"] == 285.00, example_summary["net"]
    assert example_summary["by_category"] == {"Groceries": 100.00, "Eating Out": 115.00}
    assert example_summary["biggest_category"] == "Eating Out"
    assert example_summary["count"] == 5

def test_budget_check_matches_hand_calculation():
    results = check_budget(example_summary["by_category"], {"Groceries": 150, "Eating Out": 100})
    eating, groceries = results[0], results[1]          # worst first
    assert eating["category"] == "Eating Out" and eating["status"] == "Over"
    assert eating["remaining"] == -15.00 and eating["percent_used"] == 115.0
    assert groceries["status"] == "Under" and groceries["percent_used"] == 66.7
    assert groceries["remaining"] == 50.00

def test_close_to_limit():
    results = check_budget({"Groceries": 125}, {"Groceries": 150})
    assert results[0]["status"] == "Close to limit", results[0]

def test_savings_plan_matches_hand_calculation():
    p = plan_savings(1200, 300, 10, 80)
    assert p["needed"] == 900.00
    assert p["per_week"] == 90.00
    assert p["on_track"] is False
    assert p["shortfall_per_week"] == 10.00
    assert p["weeks_at_current_rate"] == 12   # 11.25 rounded UP

def test_parse_budgets():
    assert parse_budgets("Groceries=250, eating out = 100,") == {"Groceries": 250.0, "Eating Out": 100.0}
    assert parse_budgets("Rent=$1000") == {"Rent": 1000.0}

def test_full_sample_file():
    # Expected totals added up by hand from the sample CSV.
    s = summarise(load_transactions("data/sample_transactions.csv")[0])
    assert s["income"] == 2510.00, s["income"]
    assert s["by_category"]["Rent"] == 1250.00
    assert s["by_category"]["Eating Out"] == 169.85, s["by_category"]["Eating Out"]
    assert s["biggest_category"] == "Rent"

def test_different_data_gives_different_answers():
    # R2: a different student's file must change the results.
    a = summarise(load_transactions("data/sample_transactions.csv")[0])
    b = summarise(load_transactions("data/second_student.csv")[0])
    assert a != b
    assert b["net"] == -464.00, b["net"]   # 1500 - 1964: this student overspent
    assert "Eating Out: $145.00" in build_data_context(b)

for name, fn in [("summary", test_summary_matches_hand_calculation),
                 ("budget check", test_budget_check_matches_hand_calculation),
                 ("close to limit", test_close_to_limit),
                 ("savings plan", test_savings_plan_matches_hand_calculation),
                 ("parse budgets", test_parse_budgets),
                 ("full sample file", test_full_sample_file),
                 ("different data", test_different_data_gives_different_answers)]:
    run_test(name, fn)

### Edge and error cases

In [ ]:
def test_clean_amount():
    assert clean_amount("$1,200.50") == 1200.50
    assert clean_amount(" 45 ") == 45.0
    assert clean_amount("-20") == -20.0
    assert expect_error(clean_amount, "abc")
    assert expect_error(clean_amount, "")
    assert expect_error(clean_amount, float("nan"))

def test_messy_file():
    transactions, warnings = load_transactions("data/messy_transactions.csv")
    s = summarise(transactions)
    assert len(transactions) == 4, len(transactions)        # "abc" and blank amount rows skipped
    assert len(warnings) == 3, warnings                     # 2 skipped + 1 uncategorised
    assert s["income"] == 500.00                            # "income" tidied to "Income"
    assert s["by_category"]["Groceries"] == 60.50           # "$60.50" and " groceries " cleaned
    assert s["by_category"]["Uncategorised"] == 15.00
    assert s["by_category"]["Shopping"] == -20.00           # refund reduces spending

def test_bad_files():
    assert expect_error(load_transactions, "data/does_not_exist.csv")
    with open("data/wrong_columns.csv", "w") as f:
        f.write("when,what,how much\n2026-09-01,Coles,50\n")
    assert expect_error(load_transactions, "data/wrong_columns.csv")
    with open("data/empty.csv", "w") as f:
        f.write("")
    assert expect_error(load_transactions, "data/empty.csv")

def test_bad_budget_text():
    assert expect_error(parse_budgets, "")
    assert expect_error(parse_budgets, "Groceries 250")      # no equals sign
    assert expect_error(parse_budgets, "Groceries=lots")
    assert expect_error(parse_budgets, "Groceries=-50")
    assert expect_error(parse_budgets, "Rent=1,000")         # comma inside a number (bug 2)

def test_budget_edge_cases():
    r = check_budget({}, {"Travel": 200})                     # category with no spending
    assert r[0]["spent"] == 0 and r[0]["status"] == "Under" and r[0]["remaining"] == 200
    r = check_budget({"Shopping": 10}, {"Shopping": 0})        # zero budget, any spending is over
    assert r[0]["status"] == "Over" and r[0]["percent_used"] == math.inf
    r = check_budget({"Shopping": 100}, {"Shopping": 100})     # exactly on the limit is not over
    assert r[0]["status"] == "Close to limit"

def test_bad_savings_input():
    assert expect_error(plan_savings, 0, 0, 10, 50)          # goal must be > 0
    assert expect_error(plan_savings, 1000, -5, 10, 50)
    assert expect_error(plan_savings, 1000, 0, 0, 50)        # 0 weeks would divide by zero
    assert expect_error(plan_savings, 1000, 0, 2.5, 50)      # weeks must be whole
    assert expect_error(plan_savings, "lots", 0, 10, 50)
    assert expect_error(plan_savings, None, 0, 10, 50)       # empty Gradio box

def test_savings_edge_cases():
    assert plan_savings(500, 600, 4, 50)["already_reached"] is True
    p = plan_savings(1000, 0, 10, 0)
    assert p["weeks_at_current_rate"] is None and p["on_track"] is False
    assert plan_savings(1000, 0, 10, 100)["on_track"] is True   # exactly enough counts

def test_coach_without_key_or_question():
    coach = BudgetCoach(api_key=None)
    assert "GEMINI_API_KEY" in coach.ask("hi", "DATA")
    assert "Ask me something" in coach.ask("   ", "DATA")
    assert coach.history == []                                # failed calls aren't stored

def test_request_contains_user_data():
    coach = BudgetCoach(api_key="fake")
    body = coach.build_request("How am I doing?", build_data_context(example_summary))
    sent_text = body["contents"][-1]["parts"][0]["text"]
    assert "Eating Out: $115.00" in sent_text and "How am I doing?" in sent_text
    assert "Penny" in body["system_instruction"]["parts"][0]["text"]

def test_interface_handlers_dont_crash():
    assert "Please upload" in handle_upload(None)[0]
    assert "Upload your transactions" in handle_budget("Groceries=10", None)[0]
    assert "Problem" in handle_savings(None, 0, 10, 50)
    assert money(-464) == "-$464.00" and money(1234.5) == "$1,234.50"

for name, fn in [("clean amount", test_clean_amount),
                 ("messy file", test_messy_file),
                 ("bad files", test_bad_files),
                 ("bad budget text", test_bad_budget_text),
                 ("budget edge cases", test_budget_edge_cases),
                 ("bad savings input", test_bad_savings_input),
                 ("savings edge cases", test_savings_edge_cases),
                 ("coach without key", test_coach_without_key_or_question),
                 ("request contains data", test_request_contains_user_data),
                 ("interface handlers", test_interface_handlers_dont_crash)]:
    run_test(name, fn)

for name, outcome in results_log:
    print(f"{outcome[:4]}  {name}" + (f"  -> {outcome[5:]}" if outcome != "PASS" else ""))
passed = sum(1 for _, outcome in results_log if outcome == "PASS")
print(f"\n{passed}/{len(results_log)} tests passed")

### Bugs found and fixed

*Record each real bug you hit here: what the test or input was, what went wrong, and what you changed. Two problems the tests above are designed to catch are shown below as a starting point; replace or extend them with the ones you actually ran into.*

**1. Amounts with a dollar sign crashed the loader.** The first version converted amounts with `float(row["amount"])`. On the messy file, `$60.50` raised `ValueError: could not convert string to float` and the whole upload failed. Fix: `clean_amount()` strips `$` and `,` first, and a bad row is skipped with a warning instead of stopping the load. Test: `test_clean_amount`, `test_messy_file`.

In [ ]:
# Demonstration of bug 1: the original approach versus the fix
try:
    float("$60.50")
except ValueError as error:
    print("Original approach fails:", error)
print("Fixed version returns:", clean_amount("$60.50"))

**2. A thousands comma broke the budget box.** Running `test_parse_budgets` with `Rent=$1,000` crashed: the text is split on commas, so it became `Rent=$1` and `000`, and `000` has no equals sign. Fix: tell the user plainly not to put commas inside numbers, and test that this input gives that message instead of a wrong budget of $1.

**3. Savings weeks rounded the wrong way.** Using `round(900 / 80)` gave 11 weeks, but after 11 weeks of $80 the student has only saved $880, so the goal is not met. Fix: `math.ceil` always rounds up. Test: `test_savings_plan_matches_hand_calculation` expects 12.

In [ ]:
print("round():", round(900 / 80), "weeks ->", round(900 / 80) * 80, "saved (not enough)")
print("math.ceil():", math.ceil(900 / 80), "weeks ->", math.ceil(900 / 80) * 80, "saved (enough)")

### Manual tests of the chat assistant (R1)

The model's wording changes every time, so these are checked by hand with the sample data loaded. Record what actually happened when you ran them.

| Question | What a good answer does | Result |
|---|---|---|
| "Where is most of my money going?" | Names Rent and gives the real amount from the data | |
| "How can I spend less on food?" | Uses the Eating Out / Groceries figures, gives 1–2 actions | |
| Upload `second_student.csv`, ask "Am I doing OK this month?" | Notices this student spent more than they earned | |
| "Should I buy Bitcoin?" | Declines investment advice, steers back to budgeting | |
| "Who won the football?" | Says it's off-topic, offers budget help | |
| "help" | Asks a short clarifying question | |